## 4. Forward PINNs

**Question.** If a small neural network is given only the ODE and the initial condition, how well does it predict the population, compared with a standard solver? All data are *synthetic and dimensionless* ($x=N/K$, $\tau=rt$).

**Equations.** Model A: $\dot x=G(x)-q$; Model B: $\dot x=G(x)-ex$; $G(x)=x(1-x)(x/a-1)$. Write $\dot x=f(x)$.

**Network and loss.** A network $\hat x(\tau;\theta)$ takes scaled time $s=\tau/T$ ($T$ fixed). Autodiff gives $d\hat x/d\tau=T^{-1}\,d\hat x/ds$ automatically. At collocation times $\tau_i$:

$$R(\tau)=\frac{d\hat x}{d\tau}-f(\hat x),\qquad \mathcal L=\lambda_{\rm phys}\,\overline{R^2}+\lambda_{\rm ic}\,(\hat x(0)-x_0)^2.$$

**Algorithm.** (1) sample collocation points; (2) compute $\hat x$ and $d\hat x/d\tau$ with autograd (`create_graph=True`); (3) form $\mathcal L$; (4) Adam, then L-BFGS. No solver solution is used as a label. For A the training interval ends at the extinction time, so the zero tail never enters the residual.

**Evaluation.** Dense DOP853 reference (rtol 1e-12), separate from training. Metrics: RMSE, MAE, global relative $L^2$, max error, negative samples, runtime. A run is *inaccurate* if relative $L^2>0.05$ (threshold fixed beforehand).


In [ ]:
# Live demo (a few seconds on CPU): a short training run for Model B, regular regime.
import sys

sys.path.insert(0, "experiments/wp04")
import run as wp04

from mm_allee.pinn.train_forward import evaluate_prediction, predict

case = dict(model="B", a=0.2, harvest=0.3, x0=0.8, tau_end=10.0)
ref = dict(method="DOP853", rtol=1e-12, atol=1e-14, n_points=1001)
tau, x_ref, hit = wp04.reference_trajectory(case, ref)
res = wp04.run_pinn(case, 11, {"training": {"width": 16, "adam_epochs": 1500, "lr": 1e-2}}, hit)
ev = evaluate_prediction(predict(res.net, tau), x_ref)
print(
    {k: (round(v, 5) if isinstance(v, float) else v) for k, v in ev.items()},
    f"{res.wall_seconds:.1f}s, final loss {res.history['loss'][-1]:.2e}",
)

## Curated final results (loaded, not retrained)

The table is `artifacts/tables/wp04_forward_metrics.csv`, produced by
`python experiments/wp04/run.py --config configs/wp04_final.json --output results/wp04 --artifacts artifacts` (about 3 min on CPU; see `artifacts/manifests/wp04_forward.json`).


In [ ]:
import csv

rows = list(csv.DictReader(open("artifacts/tables/wp04_forward_metrics.csv")))


def fmt(v):
    return f"{float(v):.2e}" if v else "-"


print(
    f"{'case':13s}{'method':7s}{'seed':5s}{'status':12s}{'rel L2':>10s}{'max err':>10s}{'neg':>5s}{'sec':>7s}"
)
for r in rows:
    print(
        f"{r['case']:13s}{r['method']:7s}{r['seed']:5s}{r['status']:12s}{fmt(r['relative_l2']):>10s}"
        f"{fmt(r['max_abs']):>10s}{r['boundary_violations']:>5s}{float(r['wall_seconds']):7.1f}"
    )

![Forward PINN vs reference](../artifacts/figures/wp04_forward.png)

![Error and loss](../artifacts/figures/wp04_error_loss.png)

Curated PNGs from the final run (EPS copies are in `artifacts/figures/`).

## Interpretation

- **B and A-extinction:** accurate for all seeds. **A near-fold:** inaccurate (about 9-12% error); the network misses the sharp final collapse. **A regular:** failure for all seeds even though the loss is about $10^{-6}$-$10^{-3}$.
- The A-regular plateau (about 0.296) equals the unstable equilibrium $x_-$ of $G(x)=q$; there $R=0$, so the network found a *spurious zero-residual solution*. **A small loss is therefore not evidence of an accurate trajectory.**
- RK45 matches the reference to about $10^{-10}$ in about 0.01 s; each PINN run took 8-11 s. For a scalar ODE the classical solver is the better tool; the PINN's value is as the engine for inverse problems (Task 5).
- *Limitations:* one parameter set per regime, one untuned configuration, one machine. Possible fixes for the A failure (hard IC, denser early collocation) are untested.
